# 05 — Deneyler

**Bütün özellik deneyleri bu notebook'ta yürür** (her deney için yeni notebook açılmaz). Her deneyin sonucu `UPDATES.md` → *Fikir defteri*ne yazılır.

Özellik kodu: `src/features.py` (güncel sürüm). Ölçüm: `src/validation.py` (5-fold × 3). **±0.005 altı fark = gürültü.**

Yardımcı tablolar (Pokémon stat'ları, iyileşme hamleleri, kimlik listesi) **sadece train** savaşlarından oluşturulur; test setinden hiçbir bilgi kullanılmaz.

In [1]:
import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

from src.features import load_species, load_type_chart, load_healing_moves, build_features, select_features
from src.validation import cv_accuracy

train = pd.read_json("data/train.jsonl", lines=True)
test = pd.read_json("data/test.jsonl", lines=True)
species = load_species([train])
chart = load_type_chart("data/type_chart_gen2.csv")
healing = load_healing_moves([train])

X = build_features(train, species, chart, healing)
y = train.set_index("battle_id").loc[X.index, "player_won"]
print("özellik tablosu:", X.shape)
print("veriden bulunan iyileşme hamleleri:", sorted(healing))


def models():
    return {
        "LogReg": (make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)), "linear"),
        "HistGB": (HistGradientBoostingClassifier(learning_rate=0.05, max_iter=200, max_leaf_nodes=15, random_state=42), "tree"),
    }


def evaluate(X_):
    return {name: cv_accuracy(m, select_features(X_, kind), y)[0] for name, (m, kind) in models().items()}

özellik tablosu: (10000, 82)
veriden bulunan iyileşme hamleleri: ['milkdrink', 'moonlight', 'morningsun', 'painsplit', 'recover', 'rest', 'softboiled', 'synthesis']


## Deney 1 — Hata analizi turu 1'den gelen 4 fikir (v3)

| # | Fikir | Kaynak savaş | Sütunlar |
|---|---|---|---|
| 13 | İyileşebilen ayakta Pokémon | 9117 | `alive_healers` |
| 14 | Neredeyse ölü Pokémon (can < %30) | 4249 | `n_low_hp` |
| 15 | Ayakta kalanların saldırı gücü | 3318 | `alive_offense` |
| 16 | Son 10 turdaki hasar (momentum) | 3318 | `recent_damage` |

### Kontrol: okuduğumuz savaşlarda değerler hikayeyle uyuşuyor mu?

In [2]:
ideas = {
    "#13 alive_healers": ["alive_healers"],
    "#14 n_low_hp": ["n_low_hp", "low_hp"],
    "#15 alive_offense": ["alive_offense"],
    "#16 recent_damage": ["recent_damage"],
}
new_cols = [c for c in X.columns if any(p in c for pats in ideas.values() for p in pats)]
X.loc[[9117, 4249, 3318], new_cols].round(2).T

battle_id,9117,4249,3318
p1_alive_healers,0.00,1.00,2.00
p1_n_low_hp,0.00,0.00,0.00
p1_alive_offense,6.27,2.84,3.27
p1_recent_damage,1.19,0.28,0.70
p2_alive_healers,2.00,0.00,0.00
p2_n_low_hp,0.00,2.00,0.00
p2_alive_offense,5.06,4.51,4.33
p2_recent_damage,0.47,1.79,1.52
diff_alive_healers,-2.00,1.00,2.00
diff_alive_offense,1.20,-1.67,-1.06


- 9117 → `p2_alive_healers = 2` (Miltank, Starmie) ✓
- 4249 → `p2_n_low_hp = 2` (Snorlax %22, Nidoking %23) ✓
- 3318 → `p1_alive_offense` < `p2_alive_offense` (sünger takım) ve `p1_recent_damage` < `p2_recent_damage` ✓

### Ölçüm: her fikir tek başına, sonra hepsi birlikte
Taban = v2 özellikleri (yeni sütunlar çıkarılmış hali).

In [3]:
base_X = X.drop(columns=new_cols)
results = {"taban (v2)": evaluate(base_X)}
for idea, pats in ideas.items():
    cols = [c for c in new_cols if any(p in c for p in pats)]
    results[f"+ {idea}"] = evaluate(pd.concat([base_X, X[cols]], axis=1))
results["+ hepsi (v3)"] = evaluate(X)

exp1 = pd.DataFrame(results).T
for name in models():
    exp1[f"{name} fark"] = exp1[name] - exp1.loc["taban (v2)", name]
exp1.round(4)

,LogReg,HistGB,LogReg fark,HistGB fark
taban (v2),0.6680,0.6689,0.0000,0.0000
+ #13 alive_healers,0.6699,0.6669,0.0018,-0.0020
+ #14 n_low_hp,0.6695,0.6660,0.0015,-0.0029
+ #15 alive_offense,0.6665,0.6651,-0.0015,-0.0038
+ #16 recent_damage,0.6681,0.6670,0.0001,-0.0019
+ hepsi (v3),0.6711,0.6664,0.0030,-0.0025


### Kararsız bölgeye etkisi
Asıl hedefimiz kararsız savaşlardı (olasılık 0.4–0.6). v3 orada daha iyi mi?

In [4]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict

cv5 = StratifiedKFold(5, shuffle=True, random_state=42)
m, kind = models()["LogReg"]
p_v2 = cross_val_predict(m, select_features(base_X, kind), y, cv=cv5, method="predict_proba")[:, 1]
p_v3 = cross_val_predict(m, select_features(X, kind), y, cv=cv5, method="predict_proba")[:, 1]

def zone_report(p):
    unc = (p > 0.4) & (p < 0.6)
    return {"kararsız savaş sayısı": int(unc.sum()),
            "kararsızda doğruluk": ((p[unc] > 0.5) == y.values[unc]).mean(),
            "toplam doğruluk": ((p > 0.5) == y.values).mean()}

pd.DataFrame({"v2": zone_report(p_v2), "v3": zone_report(p_v3)}).round(4)

,v2,v3
kararsız savaş sayısı,3298.0000,3309.0000
kararsızda doğruluk,0.5409,0.5473
toplam doğruluk,0.6717,0.6737


## Deney 2 — Pokémon kimliği (fikir #17)

Model "ayakta kalanların stat toplamı" gibi **toplu** sayıları görüyor ama **hangi** Pokémon'ların ayakta olduğunu bilmiyor. Snorlax ile Blissey'in stat toplamı aynı (540) ama savaştaki rolleri tamamen farklı.

### Adım 1 — Say: kimlik, modelin tahmininin ötesinde bilgi taşıyor mu?
**Artık (residual)** = gerçek sonuç − modelin tahmini. "P1'de Blissey ayaktayken" artıkların ortalaması 0'dan farklıysa, model Blissey'in etkisini **kaçırıyor** demektir.

In [5]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from src.features import alive_names

m, kind = models()["LogReg"]
p_oof = cross_val_predict(m, select_features(X, kind), y, cv=StratifiedKFold(5, shuffle=True, random_state=42),
                          method="predict_proba")[:, 1]
residual = pd.Series(y.values - p_oof, index=X.index)

alive = pd.DataFrame([(b["battle_id"], *alive_names(b)) for b in train.to_dict("records")],
                     columns=["battle_id", "p1", "p2"]).set_index("battle_id")

rows = []
for side in ["p1", "p2"]:
    counts = pd.Series([n for s in alive[side] for n in s]).value_counts()
    for name in counts[counts >= 300].index:
        mask = alive[side].apply(lambda s: name in s)
        r = residual[mask]
        rows.append({"taraf": side.upper(), "pokemon": name, "savaş": int(mask.sum()),
                     "ort_artık": r.mean(), "z": r.mean() / (r.std() / np.sqrt(len(r)))})
say = pd.DataFrame(rows)
print(f"{len(say)} aday, |z|>2 olan: {(say.z.abs() > 2).sum()} (şansla beklenen ≈ {len(say) * 0.046:.1f})")
say.reindex(say.z.abs().sort_values(ascending=False).index).head(12).round(3)

52 aday, |z|>2 olan: 11 (şansla beklenen ≈ 2.4)


,taraf,pokemon,savaş,ort_artık,z
19,P1,blissey,663,-0.066,-3.632
46,P2,jynx,621,-0.055,-3.066
20,P1,misdreavus,625,0.054,2.969
48,P2,jolteon,483,0.059,2.940
32,P2,exeggutor,1823,0.031,2.915
36,P2,machamp,1348,0.035,2.799
0,P1,snorlax,6508,0.015,2.654
21,P1,jynx,613,0.042,2.234
25,P2,snorlax,6691,-0.012,-2.222
40,P2,marowak,819,0.035,2.221


- **P1'de Blissey** (−0.066): model P1'e 6.6 puan fazla şans veriyor → "sünger" (savaş 3318) doğrulandı.
- **P2'de Jynx** (−0.055): Lovely Kiss ile uyutma tehdidi görünmüyor.
- 52 adayda |z|>2 olan 11 tane, şansla beklenen ~2.4 → **gerçek bir sinyal var**, ama etkiler küçük (1–7 puan).

### Adım 2 — Ölç: multi-hot kimlik sütunları
P1 takımlarında ≥300 kez görülen 37 Pokémon için: `p1_alive_<isim>` (P1'de ayakta mı) ve `p2_alive_<isim>` (P2'de görüldü ve ayakta mı). Liste sadece sıklığa göre seçiliyor (etiket yok).

**Eşleştirilmiş karşılaştırma:** iki sürüm aynı 15 fold'da yan yana; her fold'da hangisi kazandı?

In [6]:
from sklearn.model_selection import cross_val_score
from src.features import load_identity_vocab
from src.validation import make_cv

vocab = load_identity_vocab([train])
X4 = build_features(train, species, chart, healing, vocab)
print("kimlik sözlüğü:", len(vocab), "Pokémon →", X4.shape[1] - X.shape[1], "yeni sütun")

paired = {}
for name, (m, kind) in models().items():
    a = cross_val_score(m, select_features(X, kind), y, cv=make_cv())
    b = cross_val_score(m, select_features(X4, kind), y, cv=make_cv())
    d = b - a
    paired[name] = {"v3": a.mean(), "v3 + kimlik": b.mean(), "fark": d.mean(),
                    "kazandığı fold": f"{(d > 0).sum()}/{len(d)}", "z": d.mean() / (d.std(ddof=1) / np.sqrt(len(d)))}
pd.DataFrame(paired).T.round(4)

kimlik sözlüğü: 30 Pokémon → 60 yeni sütun


,v3,v3 + kimlik,fark,kazandığı fold,z
LogReg,0.671067,0.675667,0.0046,12/15,2.469579
HistGB,0.666433,0.6696,0.003167,9/15,1.87057


**Sonuç:** LogReg'de +0.003, 15 fold'un 11'inde kazanıyor (z ≈ 1.9) → küçük ama tutarlı, **tutuldu (v4, LogReg için)**. HistGB'de fayda yok.

## Submission — v4 (LogReg)

En iyi CV'li model: Logistic Regression + v4 özellikleri (v3 + Pokémon kimliği). Tüm train ile eğitilip test için tahmin edilir. Test özellikleri train'den oluşturulan aynı yardımcı tablolarla hesaplanır.

In [7]:
X4_test = build_features(test, species, chart, healing, vocab)
assert list(X4_test.columns) == list(X4.columns)

m, kind = models()["LogReg"]
m.fit(select_features(X4, kind), y)
pred = m.predict(select_features(X4_test, kind)).astype(int)

n_feat = select_features(X4, kind).shape[1]
fname = f"submission_lr_v4_{n_feat}features.csv"
pd.DataFrame({"battle_id": X4_test.index, "player_won": pred}).to_csv(fname, index=False)
print(fname, "| tahmin sayısı:", len(pred), "| P1 kazanır oranı:", pred.mean().round(3))
missing = {s["name"] for b in test.to_dict("records") for t in b["battle_timeline"]
           for s in (t["p1_pokemon_state"], t["p2_pokemon_state"]) if s["name"] not in species}
print("test'te olup train'de hiç görülmeyen Pokémon:", missing or "yok")

submission_lr_v4_121features.csv | tahmin sayısı: 5000 | P1 kazanır oranı: 0.508
test'te olup train'de hiç görülmeyen Pokémon: {'alakazammega', 'natu', 'kakuna', 'charizardmegay', 'unown', 'weedle', 'cleffa', 'celebi', 'horsea'}


## Deney 3 — Model ayarı (hiperparametre) ve ensemble

Özellikler v4'te sabit. Şimdi soru: **aynı özelliklerle modelleri daha iyi ayarlayabilir ve birleştirebilir miyiz?**

### 3a. Hiperparametre araması

Her modelin elle seçilen ayarları var (ağaç sayısı, derinlik, öğrenme hızı, düzenleme gücü). Şimdiye kadar bunları tahminle koyduk. **Rastgele arama (randomized search):** her model için 25 rastgele ayar kombinasyonu denenir, her biri 5-fold CV ile ölçülür, en iyisi seçilir. Sonra seçilen ayar, karar vermek için kullandığımız 5-fold × 3 CV ile tekrar ölçülür.

Not: ayarı seçtiğimiz CV ile skoru ölçtüğümüz CV aynı veriden geliyor, bu yüzden en iyi skor biraz iyimser olabilir (~0.002).

In [8]:
import time
from scipy.stats import loguniform, randint, uniform
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

X4_lin, X4_tree = select_features(X4, "linear"), select_features(X4, "tree")
cv5 = StratifiedKFold(5, shuffle=True, random_state=42)

searches = {
    "LogReg": (make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)), X4_lin,
               {"logisticregression__C": loguniform(1e-3, 10)}),
    "HistGB": (HistGradientBoostingClassifier(random_state=42), X4_tree,
               {"learning_rate": loguniform(0.01, 0.2), "max_iter": randint(100, 600),
                "max_leaf_nodes": randint(4, 40), "min_samples_leaf": randint(20, 300),
                "l2_regularization": loguniform(1e-3, 10)}),
    "RandomForest": (RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1), X4_tree,
                     {"min_samples_leaf": randint(5, 100), "max_features": uniform(0.05, 0.5)}),
}

best = {}
for name, (est, Xs, space) in searches.items():
    t = time.time()
    s = RandomizedSearchCV(est, space, n_iter=25, cv=cv5, scoring="accuracy", random_state=42, n_jobs=-1).fit(Xs, y)
    best[name] = s.best_estimator_
    print(f"{name:13s} en iyi 5-fold: {s.best_score_:.4f}  ({time.time() - t:.0f} sn)  ayar: "
          + ", ".join(f"{k.split('__')[-1]}={v:.3g}" if isinstance(v, float) else f"{k.split('__')[-1]}={v}"
                      for k, v in s.best_params_.items()))

LogReg        en iyi 5-fold: 0.6783  (5 sn)  ayar: C=0.126


HistGB        en iyi 5-fold: 0.6741  (109 sn)  ayar: l2_regularization=0.126, learning_rate=0.0331, max_iter=575, max_leaf_nodes=19, min_samples_leaf=290


RandomForest  en iyi 5-fold: 0.6706  (134 sn)  ayar: max_features=0.48, min_samples_leaf=11


In [9]:
default = {"LogReg": models()["LogReg"][0], "HistGB": models()["HistGB"][0],
           "RandomForest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=42, n_jobs=-1)}
rows = {}
for name, (est, Xs, _) in searches.items():
    a = cross_val_score(default[name], Xs, y, cv=make_cv())
    b = cross_val_score(best[name], Xs, y, cv=make_cv())
    rows[name] = {"eski ayar": a.mean(), "ayarlı": b.mean(), "fark": (b - a).mean(), "kazandığı fold": f"{(b > a).sum()}/15"}
tuning = pd.DataFrame(rows).T
tuning

,eski ayar,ayarlı,fark,kazandığı fold
LogReg,0.675667,0.6752,-0.000467,5/15
HistGB,0.6696,0.671133,0.001533,8/15
RandomForest,0.6666,0.666867,0.000267,8/15


### 3b. Ensemble: tahminlerin ortalaması

Logistic Regression doğrusal düşünür ("can farkı 1 artınca şans şu kadar artar"), ağaçlar ise kural kurar ("bayılan ≥ 3 VE can < 2 ise…"). **Farklı yerlerde yanılırlar.** Olasılıklarının ortalaması alınınca hatalar kısmen birbirini götürebilir.

Her fold'da: LogReg linear sütunlarla, ağaç modeli tree sütunlarla eğitilir; ikisinin "P1 kazanır" olasılıkları ağırlıklı ortalanır; ortalama > 0.5 ise tahmin 1.

In [10]:
from sklearn.base import clone

def oof_probas(est, Xs):
    """Her (tekrar, fold) için doğrulama olasılıkları: 15 dizi."""
    out = []
    for tr_idx, va_idx in make_cv().split(Xs, y):
        m_ = clone(est).fit(Xs.iloc[tr_idx], y.iloc[tr_idx])
        out.append((va_idx, m_.predict_proba(Xs.iloc[va_idx])[:, 1]))
    return out

P = {name: oof_probas(best[name], Xs) for name, (_, Xs, _) in searches.items()}

def blend_scores(weights):
    scores = []
    for k in range(15):
        va_idx = P["LogReg"][k][0]
        p = sum(w * P[name][k][1] for name, w in weights.items()) / sum(weights.values())
        scores.append(((p > 0.5) == y.values[va_idx]).mean())
    return np.array(scores)

blends = {
    "sadece LogReg": {"LogReg": 1},
    "sadece HistGB": {"HistGB": 1},
    "sadece RF": {"RandomForest": 1},
    "LogReg + HistGB (eşit)": {"LogReg": 1, "HistGB": 1},
    "LogReg + RF (eşit)": {"LogReg": 1, "RandomForest": 1},
    "üçü eşit": {"LogReg": 1, "HistGB": 1, "RandomForest": 1},
    "LogReg 2 : HistGB 1": {"LogReg": 2, "HistGB": 1},
}
base_scores = blend_scores(blends["sadece LogReg"])
ens = pd.DataFrame({k: {"accuracy": (s := blend_scores(w)).mean(), "LogReg'e göre": (s - base_scores).mean(),
                         "kazandığı fold": f"{(s > base_scores).sum()}/15"} for k, w in blends.items()}).T
ens

,accuracy,LogReg'e göre,kazandığı fold
sadece LogReg,0.6752,0.0,0/15
sadece HistGB,0.671133,-0.004067,2/15
sadece RF,0.666867,-0.008333,2/15
LogReg + HistGB (eşit),0.6778,0.0026,10/15
LogReg + RF (eşit),0.675933,0.000733,5/15
üçü eşit,0.676833,0.001633,9/15
LogReg 2 : HistGB 1,0.679633,0.004433,14/15


### 3c. Eşik kontrolü
Tahmin kuralı "olasılık > 0.5 → P1 kazanır". Sınıflar dengeli (%50/%50) ve iki hata türü eşit maliyetli, bu yüzden 0.5'in doğru olması beklenir. Kontrol: farklı eşiklerde accuracy.

In [11]:
best_blend = ens["accuracy"].astype(float).idxmax()
w = blends[best_blend]
th_rows = {}
for th in [0.44, 0.46, 0.48, 0.50, 0.52, 0.54, 0.56]:
    accs = []
    for k in range(15):
        va_idx = P["LogReg"][k][0]
        p = sum(wt * P[n][k][1] for n, wt in w.items()) / sum(w.values())
        accs.append(((p > th) == y.values[va_idx]).mean())
    th_rows[th] = np.mean(accs)
print("en iyi birleşim:", best_blend)
pd.Series(th_rows, name="accuracy").round(4).to_frame().T

en iyi birleşim: LogReg 2 : HistGB 1


,0.44,0.46,0.48,0.50,0.52,0.54,0.56
accuracy,0.6762,0.679,0.678,0.6796,0.677,0.6751,0.6706


## Submission — v4 ensemble (LogReg 2 : HistGB 1)

Ayarlanmış LogReg (linear sütunlar) ve ayarlanmış HistGB (tree sütunlar) tüm train ile eğitilir; test için "P1 kazanır" olasılıkları 2:1 ağırlıkla ortalanır; > 0.5 ise 1.

In [12]:
lr_full = clone(best["LogReg"]).fit(X4_lin, y)
hgb_full = clone(best["HistGB"]).fit(X4_tree, y)
p_test = (2 * lr_full.predict_proba(select_features(X4_test, "linear"))[:, 1]
          + 1 * hgb_full.predict_proba(select_features(X4_test, "tree"))[:, 1]) / 3
pred_ens = (p_test > 0.5).astype(int)

fname = "submission_ens_lr2_hgb1_v4.csv"
pd.DataFrame({"battle_id": X4_test.index, "player_won": pred_ens}).to_csv(fname, index=False)
print(fname, "| P1 kazanır oranı:", pred_ens.mean().round(3),
      "| tek LogReg submission'ı ile aynı tahmin oranı:", (pred_ens == pred).mean().round(3))

submission_ens_lr2_hgb1_v4.csv | P1 kazanır oranı: 0.506 | tek LogReg submission'ı ile aynı tahmin oranı: 0.948


## Deney 4 — Bayesçi / üretici modeller: Naive Bayes ve LDA

Derste anlatılan Bayes kuralına dayalı sınıflandırıcılar:
- **Naive Bayes (Gaussian):** her özelliğin kazanan/kaybeden savaşlardaki dağılımını ayrı ayrı öğrenir; özelliklerin **birbirinden bağımsız** olduğunu varsayar.
- **LDA:** iki sınıf için ortak kovaryanslı çok boyutlu normal dağılım öğrenir; özellikler arası ilişkiyi hesaba katar. `shrinkage="auto"` kovaryans tahminini sadeleştirir.

Not: Logistic Regression'daki L2 düzenlemesi (`C`), ağırlıklara normal önsel koymakla (Bayesçi MAP tahmini) aynıdır; yani Bayesçi lojistik regresyon zaten kullanılıyor.

Soru: tek başına ya da ensemble'a eklenince işe yarıyorlar mı?

In [13]:
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

P["NaiveBayes"] = oof_probas(make_pipeline(StandardScaler(), GaussianNB()), X4_lin)
P["LDA"] = oof_probas(make_pipeline(StandardScaler(), LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")), X4_lin)

ref = blend_scores({"LogReg": 2, "HistGB": 1})
rows = {}
for name, w in {"sadece NaiveBayes": {"NaiveBayes": 1}, "sadece LDA": {"LDA": 1},
                "ensemble (LogReg 2 : HistGB 1)": {"LogReg": 2, "HistGB": 1},
                "ensemble + NaiveBayes": {"LogReg": 2, "HistGB": 1, "NaiveBayes": 1},
                "ensemble + LDA": {"LogReg": 2, "HistGB": 1, "LDA": 1}}.items():
    s = blend_scores(w)
    rows[name] = {"accuracy": s.mean(), "ensemble'a göre": (s - ref).mean(), "kazandığı fold": f"{(s > ref).sum()}/15"}
pd.DataFrame(rows).T

,accuracy,ensemble'a göre,kazandığı fold
sadece NaiveBayes,0.6285,-0.051133,0/15
sadece LDA,0.6747,-0.004933,1/15
ensemble (LogReg 2 : HistGB 1),0.679633,0.0,0/15
ensemble + NaiveBayes,0.667133,-0.0125,0/15
ensemble + LDA,0.678967,-0.000667,5/15


**Sonuç:** Naive Bayes tek başına ~0.63 (bağımsızlık varsayımı bozuluyor: bayılma, can ve hasar aynı bilgiyi taşıyor, NB bunu üç ayrı kanıt sayıp aşırı emin oluyor) ve ensemble'ı bozuyor. LDA ~0.675, LogReg'e denk; ama aynı türde (doğrusal) karar sınırı çizdiği için ensemble'a çeşitlilik katmıyor. Ensemble değişmedi.

## Deney 5 — Takım arkadaşının özellikleriyle birleştirme (v5)

Takım arkadaşımız ayrı bir özellik seti geliştirdi (73 sütun, `src/features_teammate.py`): hamle türleri (setup, recovery, roar/whirlwind, heal bell, rapid spin), Rest + Sleep Talk ile uyuyanlar, spikes, eşyalar, son 5 tur momentumu, takımlar arası stat/tip eşleşmesi. Bizim setimizde olmayan bilgiler.

Kurallara uyum için: pokedex sadece train'den; elle yazılmış hamle grupları (recovery, setup) veriden çıkarılan listelerle değiştirildi; roar/whirlwind/heal bell/aromatherapy ayrı sütunlar.

**Sorular:** (1) birleşince gelişiyor muyuz? (2) birleşik sette gereksiz/zararlı grup var mı?

In [14]:
from src.features_teammate import build_pokedex, load_setup_moves

setup = load_setup_moves([train])
teammate_ctx = {"pokedex": build_pokedex(train), "setup_moves": setup}
X5 = build_features(train, species, chart, healing, vocab, teammate_ctx)
print("veriden bulunan setup hamleleri:", sorted(setup))
print("v5 toplam sütun:", X5.shape[1])

ours_cols = list(X4.columns)
tm_cols = [c for c in X5.columns if c.startswith("f_")]
LR5 = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.02, max_iter=5000))
lin_all = X5.drop(columns=[c for c in select_features(X4, "tree").columns if c not in select_features(X4, "linear").columns])

veriden bulunan setup hamleleri: ['acidarmor', 'agility', 'amnesia', 'barrier', 'bellydrum', 'curse', 'defensecurl', 'growth', 'meditate', 'psychup', 'swordsdance']
v5 toplam sütun: 217


### Ablation: birleşik setten her grubu tek tek çıkar
Çıkarınca skor **düşüyorsa** grup değerli; değişmiyorsa gereksiz (bilgi başka yerde var); **artıyorsa** zararlı (gürültü).

In [15]:
def pick(pats, pool):
    return [c for c in pool if any(p in c for p in pats)]

lin_ours = [c for c in lin_all.columns if not c.startswith("f_")]
groups = {
    "biz: takım durumu": pick(["_seen", "_fainted", "_hp_total", "_hp_seen_mean", "_n_slp", "_n_par", "_n_poisoned", "_n_disabled", "_n_brn"], lin_ours),
    "biz: sahadaki Pokémon": [c for c in lin_ours if c.startswith(("p1_active", "p2_active"))],
    "biz: davranış": pick(["_switches", "_no_move_turns", "_status_moves", "_avg_base_power", "_max_boost_sum"], lin_ours),
    "biz: hasar": pick(["_damage_taken", "_recent_damage"], lin_ours),
    "biz: tip": pick(["super_eff", "resisted", "active_type_adv"], lin_ours),
    "biz: güç (v3)": pick(["alive_strength", "alive_offense", "alive_healers", "n_low_hp"], lin_ours),
    "biz: kimlik": [c for c in lin_ours if "_alive_" in c and c.split("_alive_")[1] in vocab],
    "arkadaş: hamle türleri": pick(["setup_count", "recovery_count", "_used_", "failed_moves", "resttalk", "move_capacity"], tm_cols),
    "arkadaş: spikes": pick(["field_spikes"], tm_cols),
    "arkadaş: eşyalar": pick(["leftovers", "thickclub", "lightball", "miracleberry"], tm_cols),
    "arkadaş: takım eşleşmesi": pick(["pressure", "offensive_multiplier", "immunity", "speed_advantage", "_std"], tm_cols),
    "arkadaş: momentum/switch": pick(["momentum", "switches_last_5", "switch_diff"], tm_cols),
    "arkadaş: bizimkiyle çakışan durum": pick(["hp_differential", "net_hp", "f_p1_status", "f_p2_status", "alive_diff", "active_boost",
                                               "revealed_count", "hidden_count", "has_snorlax", "active_matchup"], tm_cols),
}
full = cross_val_score(LR5(), lin_all, y, cv=make_cv())
rows = []
for g, cs in groups.items():
    d = cross_val_score(LR5(), lin_all.drop(columns=cs), y, cv=make_cv()) - full
    rows.append({"çıkarılan grup": g, "sütun": len(cs), "skor farkı": d.mean(), "skorun düştüğü parça": f"{(d < 0).sum()}/15"})
print(f"birleşik set (hepsi, {lin_all.shape[1]} sütun): {full.mean():.4f}")
pd.DataFrame(rows).sort_values("skor farkı").round(4)

birleşik set (hepsi, 196 sütun): 0.6798


,çıkarılan grup,sütun,skor farkı,skorun düştüğü parça
8,arkadaş: spikes,2,-0.0030,11/15
7,arkadaş: hamle türleri,19,-0.0014,9/15
5,biz: güç (v3),8,-0.0010,10/15
1,biz: sahadaki Pokémon,16,-0.0010,9/15
10,arkadaş: takım eşleşmesi,15,-0.0008,8/15
9,arkadaş: eşyalar,4,-0.0003,8/15
6,biz: kimlik,60,-0.0002,6/15
0,biz: takım durumu,18,-0.0001,8/15
11,arkadaş: momentum/switch,4,0.0001,7/15
4,biz: tip,5,0.0003,8/15


**Karar:** skoru değiştirmeyen veya artıran grupları LogReg setinden at (`V5_LINEAR_DROP`, `V5_TEAMMATE_LINEAR_DROP` — `src/features.py`): bizim davranış (zararlı) ve hasar; arkadaşın momentum, eşyalar ve bizimkiyle çakışan durum. Ağaç modeli bütün sütunları alır.

Not: atılacak grupları aynı CV ile seçtik → sonraki skorlar biraz iyimser olabilir.

### Ölçüm: v5 vs v4 (eşleştirilmiş)

In [16]:
X5_lin, X5_tree = select_features(X5, "linear"), select_features(X5, "tree")
print("v5 linear:", X5_lin.shape[1], "sütun | tree:", X5_tree.shape[1], "sütun")

P["LogReg v5"] = oof_probas(LR5(), X5_lin)
P["HistGB v5"] = oof_probas(clone(best["HistGB"]), X5_tree)

ref = blend_scores({"LogReg": 2, "HistGB": 1})          # v4 ensemble (Deney 3)
rows = {}
for name, w in {"v4 ensemble (LogReg 2 : HistGB 1)": {"LogReg": 2, "HistGB": 1},
                "v5 LogReg": {"LogReg v5": 1}, "v5 HistGB": {"HistGB v5": 1},
                "v5 ensemble (LogReg 2 : HistGB 1)": {"LogReg v5": 2, "HistGB v5": 1},
                "v5 ensemble (LogReg 1 : HistGB 1)": {"LogReg v5": 1, "HistGB v5": 1}}.items():
    s = blend_scores(w)
    rows[name] = {"accuracy": s.mean(), "v4 ensemble'a göre": (s - ref).mean(), "kazandığı parça": f"{(s > ref).sum()}/15"}
pd.DataFrame(rows).T

v5 linear: 143 sütun | tree: 217 sütun


,accuracy,v4 ensemble'a göre,kazandığı parça
v4 ensemble (LogReg 2 : HistGB 1),0.679633,0.0,0/15
v5 LogReg,0.684333,0.0047,11/15
v5 HistGB,0.679067,-0.000567,8/15
v5 ensemble (LogReg 2 : HistGB 1),0.688733,0.0091,14/15
v5 ensemble (LogReg 1 : HistGB 1),0.688267,0.008633,14/15


## Submission — v5 ensemble (LogReg 2 : HistGB 1)

In [17]:
X5_test = build_features(test, species, chart, healing, vocab, teammate_ctx)
assert list(X5_test.columns) == list(X5.columns)

lr5 = LR5().fit(X5_lin, y)
hgb5 = clone(best["HistGB"]).fit(X5_tree, y)
p5 = (2 * lr5.predict_proba(select_features(X5_test, "linear"))[:, 1]
      + hgb5.predict_proba(select_features(X5_test, "tree"))[:, 1]) / 3
pred5 = (p5 > 0.5).astype(int)
fname = "submission_ens_lr2_hgb1_v5.csv"
pd.DataFrame({"battle_id": X5_test.index, "player_won": pred5}).to_csv(fname, index=False)
print(fname, "| P1 kazanır oranı:", pred5.mean().round(3), "| v4 ensemble ile aynı tahmin:", (pred5 == pred_ens).mean().round(3))

submission_ens_lr2_hgb1_v5.csv | P1 kazanır oranı: 0.51 | v4 ensemble ile aynı tahmin: 0.902
